In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
from google.colab import files

# ==========================================
# STEP 1: LOAD AND INSPECT THE DATA
# ==========================================
print("Downloading latest dataset from Kaggle...")
path = kagglehub.dataset_download("lakshmi25npathi/online-retail-dataset")
files_downloaded = os.listdir(path)
file_path = os.path.join(path, files_downloaded[0])

print(f"Loading data from {files_downloaded[0]} (this may take a minute)...")
df = pd.read_excel(file_path)

# CRUCIAL FIX: Strip hidden spaces from column names to prevent KeyErrors
df.columns = df.columns.str.strip()
print(f"Original shape: {df.shape}")


# ==========================================
# STEP 2: ESSENTIAL CLEANING
# ==========================================
print("Cleaning data...")
# 1. Drop rows without a Customer ID or Description
df = df.dropna(subset=['Customer ID', 'Description']).copy()

# 2. Remove cancelled orders (Invoice starts with 'C')
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# 3. Keep only strictly positive quantities and prices
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# 4. Convert Customer ID to integer (cleans up decimals like 13085.0 -> 13085)
df['Customer ID'] = df['Customer ID'].astype(int)

# 5. Convert InvoiceDate to proper datetime format
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"Cleaned shape: {df.shape}")


# ==========================================
# STEP 3: BASE FEATURE ENGINEERING
# ==========================================
# Calculate Revenue per line item
df['TotalAmount'] = df['Quantity'] * df['Price']

# Create a standard 'Month' column (setting it to the 1st of each month)
df['OrderMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()


# ==========================================
# STEP 4: THE CORE COHORT LOGIC
# ==========================================
print("Calculating Cohorts...")
# Find the first purchase month for each customer
df['CohortMonth'] = df.groupby('Customer ID')['OrderMonth'].transform('min')

# Calculate Cohort Index (Difference in months between OrderMonth and CohortMonth)
year_diff = df['OrderMonth'].dt.year - df['CohortMonth'].dt.year
month_diff = df['OrderMonth'].dt.month - df['CohortMonth'].dt.month
df['CohortIndex'] = year_diff * 12 + month_diff


# ==========================================
# STEP 5: TIME BETWEEN PURCHASES LOGIC
# ==========================================
print("Calculating Purchase Gaps...")
# Sort by Customer and Date to ensure chronological order
df = df.sort_values(['Customer ID', 'InvoiceDate'])

# Find the date of the previous order for that specific customer
df['PreviousOrderDate'] = df.groupby('Customer ID')['InvoiceDate'].shift(1)

# Calculate the gap in days
df['DaysSinceLastOrder'] = (df['InvoiceDate'] - df['PreviousOrderDate']).dt.days

# Fill NaN (for their first purchase) with 0
df['DaysSinceLastOrder'] = df['DaysSinceLastOrder'].fillna(0)


# ==========================================
# STEP 6: RFM CALCULATION AND SEGMENTATION
# ==========================================
print("Calculating and Segmenting RFM Metrics...")
# Calculate raw RFM metrics for customer_data
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1) # Reference date for Recency

customer_data = df.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda date: (snapshot_date - date.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('TotalAmount', 'sum')
).reset_index()

# 1. Score the customers into 4 tiers using quantiles
# Note: For Recency, lower days is better (so it gets a 4). For Frequency/Monetary, higher is better.
customer_data['R_Score'] = pd.qcut(customer_data['Recency'], 4, labels=[4, 3, 2, 1])
customer_data['F_Score'] = pd.qcut(customer_data['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
customer_data['M_Score'] = pd.qcut(customer_data['Monetary'], 4, labels=[1, 2, 3, 4])

# 2. Combine the scores into a single ID (e.g., "444" means top tier in all three)
customer_data['RFM_Segment_ID'] = customer_data['R_Score'].astype(str) + customer_data['F_Score'].astype(str) + customer_data['M_Score'].astype(str)

# 3. Create the text labels for Power BI
def assign_segment(rfm_id):
    if rfm_id == '444': return 'Champions'
    elif rfm_id.startswith('4') or rfm_id.startswith('3'): return 'Loyal Customers'
    elif rfm_id.startswith('1'): return 'At Risk / Churned'
    else: return 'Average / Needs Attention'

customer_data['Customer_Segment'] = customer_data['RFM_Segment_ID'].apply(assign_segment)

# 4. Add the Repeat Customer flag for your Pie Chart
customer_data['Is_Repeat_Customer'] = np.where(customer_data['Frequency'] > 1, 'Yes', 'No')

# 5. Display the first few rows to confirm it worked!
print(customer_data.head())


# ==========================================
# STEP 7: EXPORTING FOR POWER BI
# ==========================================
print("Exporting data for Power BI...")

# Drop columns we no longer need to save file size and clutter
cols_to_drop = ['Description', 'PreviousOrderDate']
df_final = df.drop(columns=cols_to_drop, errors='ignore')

# Save to CSV
df_final.to_csv("PBI_Transactions_Data.csv", index=False)
customer_data.to_csv("PBI_Customer_RFM_Data.csv", index=False)

# Trigger download to your local machine
print("Downloading files to your computer...")
files.download("PBI_Transactions_Data.csv")
files.download("PBI_Customer_RFM_Data.csv")

100%|██████████| 43.3M/43.3M [00:00<00:00, 154MB/s]

Extracting files...


Loading data from online_retail_II.xlsx (this may take a minute)...
Original shape: (525461, 8)
Cleaning data...
Cleaned shape: (407664, 8)
Calculating Cohorts...
Calculating Purchase Gaps...
Calculating and Segmenting RFM Metrics...
   Customer ID  Recency  Frequency  Monetary R_Score F_Score M_Score  \
0        12346      165         11    372.86       1       4       2   
1        12347        3          2   1323.32       4       2       3   
2        12348       74          1    222.16       2       1       1   
3        12349       43          3   2671.14       3       3       4   
4        12351       11          1    300.93       4       1       1   

  RFM_Segment_ID           Customer_Segment Is_Repeat_Customer  
0            142          At Risk / Churned                Yes  
1            423            Loyal Customers                Yes  
2            211  Average / Needs Attention                 No  
3            334            Loyal Customers                Yes  
4       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
from google.colab import files

# ==========================================
# STEP 1: LOAD AND INSPECT THE DATA
# ==========================================
print("Downloading latest dataset from Kaggle...")
path = kagglehub.dataset_download("lakshmi25npathi/online-retail-dataset")
files_downloaded = os.listdir(path)
file_path = os.path.join(path, files_downloaded[0])

print(f"Loading data from {files_downloaded[0]} (this may take a minute)...")
df = pd.read_excel(file_path)

# CRUCIAL FIX: Strip hidden spaces from column names to prevent KeyErrors
df.columns = df.columns.str.strip()
print(f"Original shape: {df.shape}")


# ==========================================
# STEP 2: ESSENTIAL CLEANING
# ==========================================
print("Cleaning data...")
# 1. Drop rows without a Customer ID or Description
df = df.dropna(subset=['Customer ID', 'Description']).copy()

# 2. Remove cancelled orders (Invoice starts with 'C')
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# 3. Keep only strictly positive quantities and prices
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# 4. Convert Customer ID to integer (cleans up decimals like 13085.0 -> 13085)
df['Customer ID'] = df['Customer ID'].astype(int)

# 5. Convert InvoiceDate to proper datetime format
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"Cleaned shape: {df.shape}")


# ==========================================
# STEP 3: BASE FEATURE ENGINEERING
# ==========================================
# Calculate Revenue per line item
df['TotalAmount'] = df['Quantity'] * df['Price']

# Create a standard 'Month' column (setting it to the 1st of each month)
df['OrderMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
